# Hermes learning-loop study: one block on one GPU

This runs one block of the pre-registered study
([`PREREGISTRATION.md`](https://github.com/BlueRingsLabs/hermes-agent/blob/learning-loop-eval/evals/learning_loop/PREREGISTRATION.md)):
Qwen3.8 27B on Hermes' pinned llama.cpp, the `carry` and `fresh` arms, then the report. Every step checks what it
uses against the pinned values and stops if anything differs.

**Colab.** Runtime, Change runtime type, **A100 GPU**, Save. Then Runtime, Run all, and keep this tab open until
the last cell finishes. Progress is copied to Google Drive every few minutes. If the session drops, or the run
stops because the model server stopped answering, Run all again: it carries on where it stopped. To stop a block
on purpose, use Runtime, Disconnect and delete runtime. The stop button only stops this page from showing the
output, while the run goes on.

**Kaggle.** Settings: Accelerator **GPU T4 x2**, Internet on, no secrets attached. Then Save Version, Save & Run All.

In [ ]:
# The block to run. Calibration first, then the confirmatory blocks the pre-registration's Amendment 1 sets.
STUDY = "calibration"
FIRST_REP, REPS = 0, 2
# The code to run: the branch for calibration, the commit named in Amendment 1 for the confirmatory run.
BRANCH, COMMIT = "learning-loop-eval", ""

In [ ]:
import os
import subprocess
import sys

REPO = "/opt/hermes-study/hermes/hermes-agent"
ON_KAGGLE = os.path.exists("/kaggle/working")


def run(*cmd):
    """Run a command, stream its output into this cell, and stop here if it fails."""
    print("$", " ".join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise SystemExit(f"stopped: exit code {process.returncode}")


def study(*args):
    run(sys.executable, f"{REPO}/evals/learning_loop/gpu_study.py", *args)


if not os.path.exists(REPO):
    run("git", "clone", "--branch", BRANCH, "https://github.com/BlueRingsLabs/hermes-agent", REPO)
if COMMIT:
    run("git", "-C", REPO, "fetch", "origin", COMMIT)
    run("git", "-C", REPO, "checkout", "--detach", COMMIT)
run("git", "-C", REPO, "log", "-1", "--format=%H %s")

In [ ]:
# Where progress is kept. On Colab that is your Drive, which the agent's account is then checked not to reach.
if ON_KAGGLE:
    CHECKPOINT, PROTECT = f"/kaggle/working/{STUDY}", []
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT, PROTECT = f"/content/drive/MyDrive/hermes-learning-loop/{STUDY}", ["/content/drive"]
os.makedirs(CHECKPOINT, exist_ok=True)
print("checkpoint:", CHECKPOINT)

In [ ]:
study("user", "--protect", *PROTECT)  # stops if the agent's account could read Drive
study("agent-env")  # Hermes' environment for that account: a few minutes the first time

In [ ]:
study("engine")  # Hermes' pinned llama.cpp, checked by sha256, or the same commit built from source
study("model")  # 16.5 GB, checked by size and sha256
study("check-preset")  # Hermes' own preset for this file must equal the pre-registered one
study("serve")  # loads the model, a minute or two

In [ ]:
study("restore", "--study", STUDY, "--checkpoint", CHECKPOINT)
study("manifest", "--study", STUDY, "--checkpoint", CHECKPOINT)
study("run", "--study", STUDY, "--checkpoint", CHECKPOINT, "--reps", str(REPS), "--first-rep", str(FIRST_REP))

In [ ]:
run(sys.executable, f"{REPO}/evals/learning_loop/report.py", f"/home/hermes-eval/runs/{STUDY}")
study("sync", "--study", STUDY, "--checkpoint", CHECKPOINT)
print("Everything is in", CHECKPOINT)